<p align="center">
  <img src="../assets/prodinno_logo.png" alt="Prodinno" width="200">
</p>

<h4 align="center">Bonus · Deep Dive</h4>
<h1 align="center">Gradient Descent, Visualized</h1>
<p align="center"><i>Watching the update rule from Session 1 actually move a line - and watching it break.</i></p>

---

## Why this notebook exists

Sessions 1's notebooks (`01_linear_regression/03_train_test_eval.ipynb` and
`02_logistic_regression/03_train_test_eval.ipynb`) derived the gradient descent update rule
algebraically:

$$\theta_j \leftarrow \theta_j - \alpha \frac{\partial J(\theta)}{\partial \theta_j}$$

That formula is correct, but it's easy to read it, nod, and never actually build an intuition
for what it *does* step by step. This notebook makes it concrete for the simplest possible
case - fitting a line $y = mx + b$ to noisy data by minimizing mean squared error - so you
can **watch** the line rotate into place, watch a too-large learning rate make it overshoot
and bounce, and drag a slider yourself to feel out the boundary between "converges," "bounces,"
and "explodes." We close by watching batch, stochastic, and mini-batch gradient descent - the
three variants introduced conceptually in Notebook 3 - take visibly different paths to the
same destination.

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

RANDOM_STATE = 7
rng = np.random.RandomState(RANDOM_STATE)

# A single feature, so the whole model is just two numbers: slope m and intercept b.
n = 60
x = rng.uniform(0, 10, n)
true_m, true_b = 2.5, 4.0
y = true_m * x + true_b + rng.normal(0, 2.0, n)

BLUE, RED, GREEN, ORANGE, PURPLE = "#4C72B0", "#C44E52", "#55A868", "#DD8452", "#8172B2"


def loss(m, b):
    '''Mean squared error of the line y = m*x + b against the data.'''
    pred = m * x + b
    return np.mean((pred - y) ** 2)


def grad(m, b, xs=None, ys=None):
    '''Gradient of the MSE loss w.r.t. (m, b), evaluated on (xs, ys) or the full dataset.'''
    xs = x if xs is None else xs
    ys = y if ys is None else ys
    pred = m * xs + b
    err = pred - ys
    gm = (2 / len(xs)) * np.sum(err * xs)
    gb = (2 / len(xs)) * np.sum(err)
    return gm, gb


# The closed-form (normal equation) solution - the exact bottom of the bowl below.
X_design = np.vstack([x, np.ones(n)]).T
theta_star, *_ = np.linalg.lstsq(X_design, y, rcond=None)
m_star, b_star = theta_star
opt_loss = loss(m_star, b_star)
print(f"True minimum:  m* = {m_star:.4f},  b* = {b_star:.4f},  loss = {opt_loss:.4f}")

True minimum:  m* = 2.5148,  b* = 4.0920,  loss = 3.2918


## 1. The Loss Surface We're Descending

$J(m, b) = \frac{1}{n}\sum_i (mx_i + b - y_i)^2$ is a function of two numbers, so we can draw
it directly as a 3D surface - a **convex bowl**, guaranteed to have exactly one minimum since
MSE is a quadratic form. Gradient descent is nothing more than standing somewhere on this
bowl and repeatedly taking a step downhill.

In [2]:
m_grid = np.linspace(-1, 7, 80)
b_grid = np.linspace(-2, 9, 80)
MM, BB = np.meshgrid(m_grid, b_grid)
Z = np.array([[loss(MM[i, j], BB[i, j]) for j in range(MM.shape[1])] for i in range(MM.shape[0])])
x_line = np.array([x.min(), x.max()])

fig_surface = go.Figure(data=[
    go.Surface(x=m_grid, y=b_grid, z=Z, colorscale="Blues", showscale=False, opacity=0.95)
])
fig_surface.add_trace(go.Scatter3d(
    x=[m_star], y=[b_star], z=[opt_loss], mode="markers",
    marker=dict(color=RED, size=6), name="minimum",
))
fig_surface.update_layout(
    title="The MSE loss surface J(m, b) - a convex bowl",
    height=500, width=750,
    scene=dict(xaxis_title="m (slope)", yaxis_title="b (intercept)", zaxis_title="loss"),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig_surface.show()

Every contour plot in the rest of this notebook is just this same bowl, viewed from
directly above (a 2D contour instead of a 3D surface) so we can clearly overlay the path
gradient descent takes across it.

## 2. Watching Gradient Descent Converge

The animation below runs batch gradient descent - the exact update rule from Session 1,
using the *entire* dataset's gradient at every step - and replays it frame by frame. **Left:**
the fitted line rotating and shifting into place over the scatter of real data. **Right:** the
same journey shown as a path crawling across the loss-surface contour toward the black ×.
Press **Play**, or drag the slider yourself.

The mechanism connecting the two panels is the whole point of this notebook: each step nudges
$(m, b)$ a little further downhill on the *right*, and that exact same nudge is what rotates
and shifts the line on the *left*. They are two views of one single update.

In [3]:
def run_gd(lr, n_iter, m0=0.0, b0=0.0, cap=1e6):
    '''Batch gradient descent. Stops early if the loss blows up (diverges).'''
    m, b = m0, b0
    hist = [(m, b, loss(m, b))]
    for _ in range(n_iter):
        gm, gb = grad(m, b)
        m -= lr * gm
        b -= lr * gb
        l = loss(m, b)
        hist.append((m, b, l))
        if not np.isfinite(l) or l > cap:
            break
    return hist


def make_convergence_figure(hist, title, line_color):
    '''An animated (Play button + slider) figure: fitted line (left) + loss-surface path (right).'''
    frames = []
    step = max(1, len(hist) // 40)
    idxs = list(range(0, len(hist), step))
    if idxs[-1] != len(hist) - 1:
        idxs.append(len(hist) - 1)

    for k in idxs:
        m_k, b_k, l_k = hist[k]
        path_m = [h[0] for h in hist[: k + 1]]
        path_b = [h[1] for h in hist[: k + 1]]
        frames.append(go.Frame(
            name=str(k),
            data=[
                go.Scatter(x=x_line, y=m_k * x_line + b_k),
                go.Scatter(x=path_m, y=path_b),
                go.Scatter(x=[m_k], y=[b_k]),
            ],
            traces=[1, 2, 3],
            layout=go.Layout(annotations=[dict(
                text=f"iter {k} · loss={l_k:.3f}", x=0.02, y=1.08,
                xref="paper", yref="paper", showarrow=False, font=dict(size=13),
            )]),
        ))

    m0, b0, l0 = hist[0]
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Best-fit line", "Loss surface (contour) + path"))
    fig.add_trace(go.Scatter(x=x, y=y, mode="markers", marker=dict(color=PURPLE, size=6, opacity=0.6), name="data"), row=1, col=1)
    fig.add_trace(go.Scatter(x=x_line, y=m0 * x_line + b0, mode="lines", line=dict(color=line_color, width=3), name="fit"), row=1, col=1)
    fig.add_trace(go.Contour(x=m_grid, y=b_grid, z=Z, colorscale="Blues", showscale=False, contours=dict(coloring="lines"), name="loss surface"), row=1, col=2)
    fig.add_trace(go.Scatter(x=[m0], y=[b0], mode="lines+markers", line=dict(color=line_color, width=2), marker=dict(size=5), name="path"), row=1, col=2)
    fig.add_trace(go.Scatter(x=[m0], y=[b0], mode="markers", marker=dict(color=line_color, size=12, line=dict(color="white", width=1)), name="current"), row=1, col=2)
    fig.add_trace(go.Scatter(x=[m_star], y=[b_star], mode="markers", marker=dict(color="black", size=10, symbol="x"), name="minimum"), row=1, col=2)

    fig.frames = frames
    fig.update_layout(
        title=title, height=480, width=980,
        annotations=[dict(text=f"iter 0 · loss={l0:.3f}", x=0.02, y=1.08, xref="paper", yref="paper", showarrow=False, font=dict(size=13))],
        updatemenus=[dict(type="buttons", showactive=False, y=1.15, x=1.0, xanchor="right", buttons=[
            dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=120, redraw=True), fromcurrent=True)]),
            dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")]),
        ])],
        sliders=[dict(steps=[dict(method="animate", args=[[str(k)], dict(mode="immediate", frame=dict(duration=0, redraw=True))], label=str(k)) for k in idxs], x=0.1, len=0.85)],
    )
    fig.update_xaxes(title_text="x", row=1, col=1)
    fig.update_yaxes(title_text="y", row=1, col=1)
    fig.update_xaxes(title_text="m (slope)", range=[m_grid[0], m_grid[-1]], row=1, col=2)
    fig.update_yaxes(title_text="b (intercept)", range=[b_grid[0], b_grid[-1]], row=1, col=2)
    return fig


GOOD_LR = 0.028
hist_good = run_gd(GOOD_LR, 80)
fig_good = make_convergence_figure(hist_good, f"Gradient descent converging (learning rate = {GOOD_LR})", BLUE)
fig_good.show()

**Reading the animation:** the line starts flat at the origin (a terrible fit) and
rotates/shifts steadily toward the data cloud. On the right, the path takes the path of
steepest descent on the contour - moving *perpendicular* to the contour lines it crosses,
exactly as the gradient (the direction of steepest increase, negated) prescribes. Progress is
fast at first (the surface is steep far from the minimum) and slows as the path approaches the
flatter bottom of the bowl - the same reason training loss curves typically drop quickly, then
level off.

## 3. When the Learning Rate Is Too High

The update rule takes a step of size $\alpha \cdot (\text{gradient})$. If $\alpha$
(the learning rate) is too large, that step **overshoots the minimum and lands on the opposite
wall of the bowl** - and the next gradient there points just as strongly back the other way.
The parameters bounce from wall to wall instead of settling, either slowly leaking energy back
toward the minimum (bounded oscillation) or gaining energy with every bounce until the values
explode toward infinity (outright divergence).

In [4]:
BOUNCE_LR = 0.031
hist_bounce = run_gd(BOUNCE_LR, 60)
fig_bounce = make_convergence_figure(
    hist_bounce, f"Too-high learning rate bouncing across the valley (learning rate = {BOUNCE_LR})", ORANGE,
)
fig_bounce.show()

**Reading the animation:** watch the right-hand path zig-zag back and forth across the
valley instead of gliding smoothly down its center - each step overshoots past the minimum,
and the next step overshoots back. It *is* still (barely) converging here, just extremely
inefficiently. Push the learning rate up only slightly further and that thin margin disappears
entirely:

In [5]:
EXPLODE_LR = 0.033
hist_explode = run_gd(EXPLODE_LR, 60)

fig_lr_compare = go.Figure()
for hist, name, color in [
    (hist_good, f"good (lr={GOOD_LR})", BLUE),
    (hist_bounce, f"bouncing (lr={BOUNCE_LR})", ORANGE),
    (hist_explode, f"diverging (lr={EXPLODE_LR})", RED),
]:
    losses = [h[2] for h in hist]
    fig_lr_compare.add_trace(go.Scatter(y=losses, mode="lines+markers", name=name, line=dict(color=color)))

fig_lr_compare.update_yaxes(type="log", title_text="loss (log scale)")
fig_lr_compare.update_xaxes(title_text="iteration")
fig_lr_compare.update_layout(title="Loss vs. iteration across learning-rate regimes", height=450, width=800)
fig_lr_compare.show()

**Reading this chart (log scale on the y-axis - note how much that compresses):** the
good learning rate's loss falls in a smooth, monotonic line. The bouncing one falls too, just
far more slowly and unevenly. The diverging one *rises* - within a couple dozen steps the loss
is already many orders of magnitude larger than where it started, headed toward infinity. In
practice this shows up as `nan` or `inf` in your loss log, and it is one of the most common
real-world bugs when hand-tuning a learning rate: **if loss is increasing or turns into `nan`,
the learning rate is very likely too high**, not the model or the data being broken.

## 4. Tool: Explore the Learning Rate Yourself

Drag the slider below across eleven learning rates spanning "far too cautious" to "far too
aggressive," and watch both panels update: the resulting best-fit line (left) after a fixed 80
steps, and the actual parameter path taken to get there (right). The title reports the final
loss and a plain-language verdict for whichever learning rate is selected.

In [6]:
LR_VALUES = [0.002, 0.008, 0.015, 0.02, 0.025, 0.028, 0.0305, 0.031, 0.0315, 0.032, 0.033]
N_ITER = 80

fig_tool = make_subplots(rows=1, cols=2, subplot_titles=("Best-fit line after 80 steps", "Parameter path on the loss surface"))
fig_tool.add_trace(go.Scatter(x=x, y=y, mode="markers", marker=dict(color=PURPLE, size=6, opacity=0.6), name="data", showlegend=False), row=1, col=1)
fig_tool.add_trace(go.Contour(x=m_grid, y=b_grid, z=Z, colorscale="Blues", showscale=False, contours=dict(coloring="lines")), row=1, col=2)
fig_tool.add_trace(go.Scatter(x=[m_star], y=[b_star], mode="markers", marker=dict(color="black", size=10, symbol="x"), name="minimum"), row=1, col=2)

N_BASE = 3
steps = []
for i, lr in enumerate(LR_VALUES):
    hist = run_gd(lr, N_ITER)
    final_m, final_b, final_l = hist[-1]
    diverged = not np.isfinite(final_l) or final_l > 1000

    m_seq = np.array([h[0] for h in hist])
    deltas = np.diff(m_seq)
    sign_changes = np.sum(np.diff(np.sign(deltas)) != 0) if len(deltas) > 1 else 0
    oscillating = sign_changes >= 0.5 * len(deltas)

    if diverged:
        status, color = "DIVERGED - parameters shot off to infinity", RED
    elif oscillating:
        status, color = "BOUNCING - overshooting the minimum every step", ORANGE
    elif final_l > 1.3 * opt_loss:
        status, color = "SLOW - still converging, hasn't reached the minimum yet", BLUE
    else:
        status, color = "CONVERGED smoothly", GREEN

    path_m = np.clip([h[0] for h in hist], m_grid[0], m_grid[-1])
    path_b = np.clip([h[1] for h in hist], b_grid[0], b_grid[-1])

    if diverged:
        fig_tool.add_trace(go.Scatter(x=x_line, y=[np.nan, np.nan], mode="lines", line=dict(color=color, width=3), visible=False, showlegend=False), row=1, col=1)
    else:
        fig_tool.add_trace(go.Scatter(x=x_line, y=final_m * x_line + final_b, mode="lines", line=dict(color=color, width=3), visible=False, showlegend=False), row=1, col=1)

    fig_tool.add_trace(go.Scatter(
        x=path_m, y=path_b, mode="lines+markers", line=dict(color=color, width=2),
        marker=dict(size=4, color=list(range(len(path_m))), colorscale="Greys"),
        visible=False, showlegend=False,
    ), row=1, col=2)
    fig_tool.add_trace(go.Scatter(
        x=[path_m[-1]], y=[path_b[-1]], mode="markers",
        marker=dict(color=color, size=13, line=dict(color="white", width=1)),
        visible=False, showlegend=False,
    ), row=1, col=2)

    vis = [True, True, True] + [False] * (3 * len(LR_VALUES))
    vis[N_BASE + 3 * i: N_BASE + 3 * i + 3] = [True, True, True]
    steps.append(dict(
        method="update",
        args=[{"visible": vis}, {"title": f"learning rate = {lr}  -  final loss = {final_l:.3g}  -  {status}"}],
        label=str(lr),
    ))

DEFAULT_IDX = LR_VALUES.index(0.02)
for i in range(len(LR_VALUES)):
    on = (i == DEFAULT_IDX)
    fig_tool.data[N_BASE + 3 * i].visible = on
    fig_tool.data[N_BASE + 3 * i + 1].visible = on
    fig_tool.data[N_BASE + 3 * i + 2].visible = on

fig_tool.update_layout(
    height=480, width=1000,
    title=steps[DEFAULT_IDX]["args"][1]["title"],
    sliders=[dict(active=DEFAULT_IDX, currentvalue=dict(prefix="learning rate: "), steps=steps, x=0.1, len=0.85)],
)
fig_tool.update_yaxes(range=[y.min() - 15, y.max() + 15], row=1, col=1)
fig_tool.update_xaxes(range=[m_grid[0], m_grid[-1]], row=1, col=2)
fig_tool.update_yaxes(range=[b_grid[0], b_grid[-1]], row=1, col=2)
fig_tool.show()

**What to notice as you drag the slider from left to right:** the smallest learning
rates are perfectly stable but haven't gotten anywhere near the data in 80 steps ("slow"); a
middle band converges cleanly; right at the edge the path starts visibly bouncing; and just
past that edge, the fit line panel goes blank - the parameters have flown off the chart
entirely. There is no universal "correct" learning rate; it depends on the scale of your data
and features, which is exactly why **feature scaling** (`StandardScaler`, used throughout this
workshop) matters: it keeps every feature's gradient on a comparable scale, so one single
learning rate works reasonably well across all of them at once.

## 5. Batch vs. Stochastic vs. Mini-Batch Gradient Descent

Notebook 3 introduced these three variants conceptually. Here they are, run on the exact same
data, so their trajectories can be compared directly:

- **Batch** - one update per epoch, using the gradient over *all* 60 points.
- **Stochastic (SGD)** - one update per *single* random point, 60 updates per epoch.
- **Mini-batch** - one update per random batch of 8 points, ~8 updates per epoch.

In [7]:
def run_sgd(lr, n_epochs, seed=0, m0=0.0, b0=0.0):
    rs = np.random.RandomState(seed)
    m, b = m0, b0
    path = [(m, b)]
    epoch_loss = [loss(m, b)]
    idx = np.arange(n)
    for _ in range(n_epochs):
        rs.shuffle(idx)
        for i in idx:
            gm, gb = grad(m, b, x[i:i + 1], y[i:i + 1])
            m -= lr * gm
            b -= lr * gb
            path.append((m, b))
        epoch_loss.append(loss(m, b))
    return path, epoch_loss


def run_minibatch(lr, n_epochs, batch_size=8, seed=0, m0=0.0, b0=0.0):
    rs = np.random.RandomState(seed)
    m, b = m0, b0
    path = [(m, b)]
    epoch_loss = [loss(m, b)]
    idx = np.arange(n)
    for _ in range(n_epochs):
        rs.shuffle(idx)
        for start in range(0, n, batch_size):
            batch_idx = idx[start:start + batch_size]
            gm, gb = grad(m, b, x[batch_idx], y[batch_idx])
            m -= lr * gm
            b -= lr * gb
            path.append((m, b))
        epoch_loss.append(loss(m, b))
    return path, epoch_loss


N_EPOCHS = 25
hist_batch_ep = run_gd(0.028, N_EPOCHS)
path_batch = [(h[0], h[1]) for h in hist_batch_ep]
el_batch = [h[2] for h in hist_batch_ep]
path_sgd, el_sgd = run_sgd(0.01, N_EPOCHS)
path_mb, el_mb = run_minibatch(0.015, N_EPOCHS, batch_size=8)

print(f"Updates per epoch  -  batch: 1,  stochastic: {n},  mini-batch: {int(np.ceil(n / 8))}")
print(f"Loss after {N_EPOCHS} epochs  -  batch: {el_batch[-1]:.4f},  stochastic: {el_sgd[-1]:.4f},  mini-batch: {el_mb[-1]:.4f}")

Updates per epoch  -  batch: 1,  stochastic: 60,  mini-batch: 8
Loss after 25 epochs  -  batch: 4.8120,  stochastic: 3.4321,  mini-batch: 3.5373


In [8]:
fig_types = make_subplots(rows=1, cols=2, subplot_titles=("Parameter path on the loss surface", "Loss per epoch (evaluated on the full dataset)"))
fig_types.add_trace(go.Contour(x=m_grid, y=b_grid, z=Z, colorscale="Blues", showscale=False, contours=dict(coloring="lines")), row=1, col=1)

for path, name, color, subsample in [
    (path_batch, "Batch (1 update/epoch)", BLUE, 1),
    (path_sgd, "Stochastic (1 sample/update)", RED, 5),
    (path_mb, "Mini-batch (8 samples/update)", GREEN, 2),
]:
    pm = [p[0] for p in path][::subsample]
    pb = [p[1] for p in path][::subsample]
    fig_types.add_trace(go.Scatter(x=pm, y=pb, mode="lines+markers", name=name, line=dict(color=color, width=1.2), marker=dict(size=3, color=color)), row=1, col=1)

fig_types.add_trace(go.Scatter(x=[m_star], y=[b_star], mode="markers", marker=dict(color="black", size=11, symbol="x"), name="minimum"), row=1, col=1)

for el, name, color in [(el_batch, "Batch", BLUE), (el_sgd, "Stochastic", RED), (el_mb, "Mini-batch", GREEN)]:
    fig_types.add_trace(go.Scatter(x=list(range(len(el))), y=el, mode="lines+markers", name=name, line=dict(color=color), showlegend=False), row=1, col=2)

fig_types.update_yaxes(type="log", title_text="loss on full dataset (log scale)", row=1, col=2)
fig_types.update_xaxes(title_text="epoch", row=1, col=2)
fig_types.update_xaxes(title_text="m (slope)", range=[m_grid[0], m_grid[-1]], row=1, col=1)
fig_types.update_yaxes(title_text="b (intercept)", range=[b_grid[0], b_grid[-1]], row=1, col=1)
fig_types.update_layout(title=f"Batch vs. stochastic vs. mini-batch gradient descent (same {N_EPOCHS} epochs)", height=480, width=1050, legend=dict(orientation="h", y=-0.15))
fig_types.show()

**Reading the comparison:** the batch path (blue) is perfectly smooth - every step is
the exact steepest-descent direction for the whole dataset. The stochastic path (red) is a
visibly noisy zig-zag - each step follows only one point's gradient, which rarely points
exactly toward the true minimum, yet the *overall drift* still heads the right way. Mini-batch
(green) sits in between: noisier than batch, smoother than pure SGD.

Despite the noise, stochastic and mini-batch **reach a lower loss in the same 25 epochs** than
batch does. This is the practical reason SGD dominates at scale: an "epoch" for batch GD buys
exactly one update, while the same epoch buys 60 updates for SGD and ~8 for mini-batch - far
more chances to move downhill, even if each individual step is a rougher estimate of the true
gradient. This is precisely the trade-off table from Notebook 3, Section 1.5, made visible.

## Summary

- Watched batch gradient descent visibly rotate a line into a best fit, with the fitted-line
  panel and the loss-surface-path panel shown as two views of the exact same update.
- Saw a too-high learning rate cause the parameter path to bounce between the walls of the
  loss bowl, and a slightly-higher learning rate turn that bounce into outright divergence
  (loss rising toward infinity instead of falling) - and connected `nan`/rising loss in
  practice to "the learning rate is probably too high."
- Used an interactive slider tool to explore eleven learning rates directly, watching the same
  model, run for the same number of steps, transition from *slow* → *converged* →
  *bouncing* → *diverged*.
- Compared batch, stochastic, and mini-batch gradient descent on identical data: a smooth
  path taking few, exact steps vs. noisy paths taking many, cheap, approximate steps - and
  saw the noisy variants win on loss after the same number of epochs, because they simply take
  far more update steps per epoch.